<a href="https://colab.research.google.com/github/davidrpugh/machine-learning-for-tabular-data/blob/main/notebooks/lecture-4-part-1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import compose, datasets, linear_model, metrics, model_selection
from sklearn import preprocessing, pipeline, utils

# Polynomial Regression with Scikit-Learn

In [2]:
features, targets = datasets.load_diabetes(
    return_X_y=True,
    as_frame=True,
    scaled=False
)

## Train-test split

In [3]:
prng = np.random.RandomState(42)

train_features, test_features, train_targets, test_targets = (
    model_selection.train_test_split(
        features,
        targets,
        random_state=prng,
        test_size=0.2
    )
)

## Features Preprocessing

### Feature Encoding

In [18]:
feature_encoding = compose.make_column_transformer(
    (
        preprocessing.OneHotEncoder(
            drop="first",
            sparse_output=False,
        ),
        ["sex"]
    ),
    force_int_remainder_cols=False,
    remainder="passthrough",
    verbose=False,
    verbose_feature_names_out=False
).set_output(transform="pandas")


In [19]:
encoded_train_features = feature_encoding.fit_transform(
    train_features,
    train_targets
)

In [20]:
encoded_train_features

,sex_2.0,age,bmi,bp,s1,s2,s3,s4,s5,s6
17,1.0,68.0,27.5,111.0,214.0,147.0,39.0,5.0,4.9416,91.0
66,1.0,46.0,24.7,85.0,174.0,123.2,30.0,6.0,4.6444,96.0
137,0.0,50.0,31.0,123.0,178.0,105.0,48.0,4.0,4.8283,88.0
245,0.0,41.0,23.1,86.0,148.0,78.0,58.0,3.0,4.0943,60.0
31,0.0,42.0,20.3,71.0,161.0,81.2,66.0,2.0,4.2341,81.0
...,...,...,...,...,...,...,...,...,...,...
106,0.0,22.0,19.3,82.0,156.0,93.2,52.0,3.0,3.9890,71.0
270,1.0,50.0,29.2,119.0,162.0,85.2,54.0,3.0,4.7362,95.0
348,0.0,57.0,24.5,93.0,186.0,96.6,71.0,3.0,4.5218,91.0
435,0.0,45.0,24.2,83.0,177.0,118.4,45.0,4.0,4.2195,82.0


### Feature Engineering

In [24]:
feature_engineering = preprocessing.PolynomialFeatures(
    degree=2,
    include_bias=False,
    interaction_only=False
).set_output(transform="pandas")

In [25]:
engineered_train_features = feature_engineering.fit_transform(
    encoded_train_features,
    train_targets
)

In [26]:
engineered_train_features

,sex_2.0,age,bmi,bp,s1,s2,s3,s4,s5,s6,...,s3^2,s3 s4,s3 s5,s3 s6,s4^2,s4 s5,s4 s6,s5^2,s5 s6,s6^2
17,1.0,68.0,27.5,111.0,214.0,147.0,39.0,5.0,4.9416,91.0,...,1521.0,195.0,192.7224,3549.0,25.0,24.7080,455.0,24.419411,449.6856,8281.0
66,1.0,46.0,24.7,85.0,174.0,123.2,30.0,6.0,4.6444,96.0,...,900.0,180.0,139.3320,2880.0,36.0,27.8664,576.0,21.570451,445.8624,9216.0
137,0.0,50.0,31.0,123.0,178.0,105.0,48.0,4.0,4.8283,88.0,...,2304.0,192.0,231.7584,4224.0,16.0,19.3132,352.0,23.312481,424.8904,7744.0
245,0.0,41.0,23.1,86.0,148.0,78.0,58.0,3.0,4.0943,60.0,...,3364.0,174.0,237.4694,3480.0,9.0,12.2829,180.0,16.763292,245.6580,3600.0
31,0.0,42.0,20.3,71.0,161.0,81.2,66.0,2.0,4.2341,81.0,...,4356.0,132.0,279.4506,5346.0,4.0,8.4682,162.0,17.927603,342.9621,6561.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,22.0,19.3,82.0,156.0,93.2,52.0,3.0,3.9890,71.0,...,2704.0,156.0,207.4280,3692.0,9.0,11.9670,213.0,15.912121,283.2190,5041.0
270,1.0,50.0,29.2,119.0,162.0,85.2,54.0,3.0,4.7362,95.0,...,2916.0,162.0,255.7548,5130.0,9.0,14.2086,285.0,22.431590,449.9390,9025.0
348,0.0,57.0,24.5,93.0,186.0,96.6,71.0,3.0,4.5218,91.0,...,5041.0,213.0,321.0478,6461.0,9.0,13.5654,273.0,20.446675,411.4838,8281.0
435,0.0,45.0,24.2,83.0,177.0,118.4,45.0,4.0,4.2195,82.0,...,2025.0,180.0,189.8775,3690.0,16.0,16.8780,328.0,17.804180,345.9990,6724.0


### Feature Scaling

In [27]:
feature_scaling = compose.make_column_transformer(
    (
        "passthrough",
        ["sex_2.0"]
    ),
    force_int_remainder_cols=False,
    remainder=preprocessing.StandardScaler(),
    verbose=False,
    verbose_feature_names_out=False
).set_output(transform="pandas")


In [28]:
scaled_train_features = feature_scaling.fit_transform(
    engineered_train_features,
    train_targets
)

In [29]:
scaled_train_features

,sex_2.0,age,bmi,bp,s1,s2,s3,s4,s5,s6,...,s3^2,s3 s4,s3 s5,s3 s6,s4^2,s4 s5,s4 s6,s5^2,s5 s6,s6^2
17,1.0,1.498365,0.219902,1.138874,0.728473,1.055893,-0.824451,0.711038,0.547482,-0.061449,...,-0.763865,0.139814,-0.643782,-0.833337,0.554248,0.680491,0.494405,0.497357,0.223538,-0.123122
66,1.0,-0.228858,-0.419366,-0.710591,-0.424929,0.272425,-1.529791,1.484286,-0.019757,0.367236,...,-1.197456,-0.271960,-1.617757,-1.420244,1.467286,1.086102,1.300126,-0.074412,0.180462,0.308045
137,0.0,0.085182,1.018987,1.992473,-0.309589,-0.326699,-0.119111,-0.062210,0.331237,-0.318660,...,-0.217163,0.057459,0.068333,-0.241166,-0.192782,-0.012325,-0.191457,0.275203,-0.055829,-0.370754
245,0.0,-0.621409,-0.784662,-0.639458,-1.174640,-1.215508,0.664600,-0.835458,-1.069682,-2.719299,...,0.522944,-0.436670,0.172516,-0.893870,-0.773806,-0.915177,-1.336780,-1.039180,-2.075231,-2.281721
31,0.0,-0.542899,-1.423930,-1.706457,-0.799784,-1.110167,1.291569,-1.608706,-0.802859,-0.918820,...,1.215573,-1.589637,0.938358,0.743152,-1.188823,-1.405072,-1.456639,-0.805510,-0.978910,-0.916284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,-2.113102,-1.652240,-0.923991,-0.943960,-0.715141,0.194374,-0.835458,-1.270659,-1.776191,...,0.062123,-0.930799,-0.375515,-0.707884,-0.773806,-0.955746,-1.117037,-1.210005,-1.652033,-1.617217
270,1.0,0.085182,0.608029,1.707940,-0.770949,-0.978492,0.351116,-0.835458,0.155454,0.281499,...,0.210144,-0.766089,0.506087,0.553658,-0.773806,-0.667873,-0.637600,0.098414,0.226393,0.219967
348,0.0,0.634754,-0.465028,-0.141525,-0.078908,-0.603217,1.683425,-0.835458,-0.253752,-0.061449,...,1.693849,0.633943,1.697196,1.721330,-0.773806,-0.750475,-0.717506,-0.299947,-0.206880,-0.123122
435,0.0,-0.307368,-0.533521,-0.852858,-0.338424,0.114414,-0.354224,-0.062210,-0.830724,-0.833083,...,-0.411965,-0.271960,-0.695680,-0.709639,-0.192782,-0.325061,-0.351269,-0.830280,-0.944694,-0.841118


### Creating a Pipeline

In [32]:
feature_preprocessing = pipeline.make_pipeline(
    feature_encoding,
    feature_engineering,
    feature_scaling,
).set_output(transform="pandas")

In [33]:
preprocessed_train_features = feature_preprocessing.fit_transform(
    train_features,
    train_targets
)

In [34]:
preprocessed_train_features

,sex_2.0,age,bmi,bp,s1,s2,s3,s4,s5,s6,...,s3^2,s3 s4,s3 s5,s3 s6,s4^2,s4 s5,s4 s6,s5^2,s5 s6,s6^2
17,1.0,1.498365,0.219902,1.138874,0.728473,1.055893,-0.824451,0.711038,0.547482,-0.061449,...,-0.763865,0.139814,-0.643782,-0.833337,0.554248,0.680491,0.494405,0.497357,0.223538,-0.123122
66,1.0,-0.228858,-0.419366,-0.710591,-0.424929,0.272425,-1.529791,1.484286,-0.019757,0.367236,...,-1.197456,-0.271960,-1.617757,-1.420244,1.467286,1.086102,1.300126,-0.074412,0.180462,0.308045
137,0.0,0.085182,1.018987,1.992473,-0.309589,-0.326699,-0.119111,-0.062210,0.331237,-0.318660,...,-0.217163,0.057459,0.068333,-0.241166,-0.192782,-0.012325,-0.191457,0.275203,-0.055829,-0.370754
245,0.0,-0.621409,-0.784662,-0.639458,-1.174640,-1.215508,0.664600,-0.835458,-1.069682,-2.719299,...,0.522944,-0.436670,0.172516,-0.893870,-0.773806,-0.915177,-1.336780,-1.039180,-2.075231,-2.281721
31,0.0,-0.542899,-1.423930,-1.706457,-0.799784,-1.110167,1.291569,-1.608706,-0.802859,-0.918820,...,1.215573,-1.589637,0.938358,0.743152,-1.188823,-1.405072,-1.456639,-0.805510,-0.978910,-0.916284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,-2.113102,-1.652240,-0.923991,-0.943960,-0.715141,0.194374,-0.835458,-1.270659,-1.776191,...,0.062123,-0.930799,-0.375515,-0.707884,-0.773806,-0.955746,-1.117037,-1.210005,-1.652033,-1.617217
270,1.0,0.085182,0.608029,1.707940,-0.770949,-0.978492,0.351116,-0.835458,0.155454,0.281499,...,0.210144,-0.766089,0.506087,0.553658,-0.773806,-0.667873,-0.637600,0.098414,0.226393,0.219967
348,0.0,0.634754,-0.465028,-0.141525,-0.078908,-0.603217,1.683425,-0.835458,-0.253752,-0.061449,...,1.693849,0.633943,1.697196,1.721330,-0.773806,-0.750475,-0.717506,-0.299947,-0.206880,-0.123122
435,0.0,-0.307368,-0.533521,-0.852858,-0.338424,0.114414,-0.354224,-0.062210,-0.830724,-0.833083,...,-0.411965,-0.271960,-0.695680,-0.709639,-0.192782,-0.325061,-0.351269,-0.830280,-0.944694,-0.841118


## Target Preprocessing

In [6]:
target_preprocessor = preprocessing.FunctionTransformer(
    func=np.log,
    inverse_func=np.exp
)

In [7]:
target_preprocessor

FunctionTransformer(func=<ufunc 'log'>, inverse_func=<ufunc 'exp'>)

## Model training

### Using LinearRegression

In [35]:
_regressor = compose.TransformedTargetRegressor(
    regressor=linear_model.LinearRegression(),
    transformer=target_preprocessor
)

linear_regression_pipeline = pipeline.make_pipeline(
    feature_encoding,
    feature_engineering,
    feature_scaling,
    _regressor
)

In [36]:
linear_regression_pipeline

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(force_int_remainder_cols=False,
                                   remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['sex'])],
                                   verbose_feature_names_out=False)),
                ('polynomialfeatures', PolynomialFeatures(include_bias=False)),
                ('columntransformer-2',
                 ColumnTransformer(force_int_remainder_cols=False,
                                   remainder=StandardScaler(),
                                   transformers=[('passthrough', 'passthrough',
                                                  ['sex_2.0'])],
                                   verbose_feature_names_out=False)),
                ('transformedtargetregressor',
                 TransformedTargetRegressor(regressor=LinearRegression(),
                                            transformer=FunctionTransformer(func=<ufunc 'log'>,
                                                                            inverse_func=<ufunc 'exp'>)))])

In [37]:
%%timeit
_ = linear_regression_pipeline.fit(train_features, train_targets)

26.6 ms ± 1.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [38]:
_ = linear_regression_pipeline.fit(train_features, train_targets)

In [39]:
train_predictions = linear_regression_pipeline.predict(train_features)
train_rmse = metrics.root_mean_squared_error(
    train_targets,
    train_predictions,
)
print(f"Training rmse: {train_rmse}")

Training rmse: 51.56154486609228


In [40]:
test_predictions = linear_regression_pipeline.predict(test_features)
test_rmse = metrics.root_mean_squared_error(
    test_targets,
    test_predictions,
)
print(f"Testing rmse: {test_rmse}")

Testing rmse: 70.41583425172277


### Using SGDRegressor

In [46]:
_regressor = compose.TransformedTargetRegressor(
    regressor=linear_model.SGDRegressor(
        alpha=0.0
    ),
    transformer=target_preprocessor
)

sgd_regressor_pipeline = pipeline.make_pipeline(
    feature_encoding,
    feature_engineering,
    feature_scaling,
    _regressor
)

In [48]:
%%timeit
_ = sgd_regressor_pipeline.fit(train_features, train_targets)

29.7 ms ± 6.74 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [49]:
_ = sgd_regressor_pipeline.fit(train_features, train_targets)

In [50]:
train_predictions = sgd_regressor_pipeline.predict(train_features)
train_rmse = metrics.root_mean_squared_error(
    train_targets,
    train_predictions,
)
print(f"Training rmse: {train_rmse}")

Training rmse: 58.374251900011785


In [51]:
test_predictions = sgd_regressor_pipeline.predict(test_features)
test_rmse = metrics.root_mean_squared_error(
    test_targets,
    test_predictions,
)
print(f"Testing rmse: {test_rmse}")

Testing rmse: 61.54165046738852


### Exercise

Compare the training loss and the testing loss. Is the model underfitting or overfitting? How can you tell?